# Notebook 04: TrOCR Fine-Tuning & Domain Adaptation on OFPHIL Dataset
**Phase:** 05 Document Processing and OCR  
**Thesis:** A Multimodal Machine Learning Approach Integrating CNN and Transformer Architectures for Charity Patient Eligibility Classification  
**Institution:** Ophthalmological Foundation of the Philippines (OFPHIL)  

---
### 1. Purpose and Objectives
Fine-tunes the Vision Transformer encoder-decoder architecture (`microsoft/trocr-base-handwritten`) on verified OFPHIL clinical, billing, and administrative text crops. Integrates Domain-Specific Lexicon post-processing to target $\ge 90\%$ Character Accuracy ($\le 10\%$ CER).

In [1]:
# Step 1: Environment Setup, GPU Verification & Clean Console
import os
import sys
import warnings
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# Clean console: Suppress non-critical PyTorch / HuggingFace parameter warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', message='.*lr_scheduler.step().*')
import logging
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

PROJECT_ROOT = [p for p in [Path.cwd()] + list(Path.cwd().parents) if p.name == 'dss-charity-eligibity-classification'][0]
DATA_DIR = PROJECT_ROOT / 'data'
INTERIM_DIR = DATA_DIR / 'interim' / 'document_processing'
STD_DIR = INTERIM_DIR / 'standardized'
MODEL_SAVE_DIR = PROJECT_ROOT / 'models' / 'trocr_ofphil_finetuned'
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('='*60)
print(' OFPHIL TrOCR Fine-Tuning Environment Setup')
print('='*60)
print(f'Execution Device   : {device}')
if device.type == 'cuda':
    print(f'GPU Device Name    : {torch.cuda.get_device_name(0)}')
    print(f'Initial VRAM Alloc : {torch.cuda.memory_allocated(0)/(1024**2):.1f} MB')
print(f'Project Root       : {PROJECT_ROOT}')
print(f'Model Output Dir   : {MODEL_SAVE_DIR}')
print('='*60)

 OFPHIL TrOCR Fine-Tuning Environment Setup
Execution Device   : cuda
GPU Device Name    : NVIDIA GeForce RTX 3050 Laptop GPU
Initial VRAM Alloc : 0.0 MB
Project Root       : c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\dss-charity-eligibity-classification
Model Output Dir   : c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\dss-charity-eligibity-classification\models\trocr_ofphil_finetuned


In [2]:
# Step 2: Load Dataset & Filter Valid Text Crops
import re

# Load latest ground-truth annotations directly to include all newly pulled flatbeds and case scans
gt_ocr_p = PROJECT_ROOT / 'data' / 'interim' / 'document_processing' / 'ground_truth_annotations' / 'ocr_transcription_annotations.csv'
gt_det_p = PROJECT_ROOT / 'data' / 'interim' / 'document_processing' / 'ground_truth_annotations' / 'text_detection_annotations.csv'

if not gt_ocr_p.exists():
    gt_ocr_p = STD_DIR / 'ocr_transcription_annotations.csv'
    gt_det_p = STD_DIR / 'text_detection_annotations.csv'

df_det = pd.read_csv(gt_det_p)
df_ocr = pd.read_csv(gt_ocr_p)

merged_df = pd.merge(
    df_ocr[['file_name', 'box_id', 'box_type', 'text_modality', 'ground_truth_transcription', 'relative_path']],
    df_det[['file_name', 'box_id', 'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height']],
    on=['file_name', 'box_id'],
    how='inner'
)

# Robust text cleaner: exclude pure symbols, empty checkboxes, non-character blanks
def is_valid_text(val):
    if pd.isna(val): return False
    s = str(val).strip()
    if not s or s.upper() in ['[NONE]', '[ ]', '[X]', '[✓]', 'NONE']:
        return False
    return bool(re.search(r'[a-zA-Z0-9]', s))

valid_df = merged_df[
    (~merged_df['box_type'].isin(['SIGNATURE', 'TABLE_CELL'])) &
    (merged_df['ground_truth_transcription'].apply(is_valid_text)) &
    (merged_df['bbox_width'] > 8) &
    (merged_df['bbox_height'] > 8)
].copy().reset_index(drop=True)

print(f'Total Valid Text Samples for Fine-Tuning: {len(valid_df)}')
print('Distribution by Modality:')
print(valid_df['box_type'].value_counts())

Total Valid Text Samples for Fine-Tuning: 21888
Distribution by Modality:
box_type
FORM_SECTION             13882
HANDWRITTEN_ENTRY         6799
STAMP_OR_SEAL              592
CHECKMARK_OR_CHECKBOX      477
signature                  104
handwritten                 28
printed                      6
Name: count, dtype: int64


In [3]:
# Step 3: PyTorch Dataset for Cropped Text Images
class OFPHILOCRDataset(Dataset):
    def __init__(self, df, project_root, processor, max_target_length=64):
        self.df = df
        self.project_root = project_root
        self.processor = processor
        self.max_target_length = max_target_length
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.project_root / row['relative_path']
        
        bx, by = int(row['bbox_x']), int(row['bbox_y'])
        bw, bh = int(row['bbox_width']), int(row['bbox_height'])
        
        try:
            with Image.open(img_path).convert('RGB') as img:
                crop = img.crop((bx, by, bx + bw, by + bh))
                pixel_values = self.processor(crop, return_tensors='pt').pixel_values.squeeze()
        except Exception:
            pixel_values = torch.zeros((3, 384, 384))
            
        target_text = str(row['ground_truth_transcription']).strip()
        labels = self.processor.tokenizer(
            target_text,
            padding='max_length',
            max_length=self.max_target_length,
            truncation=True,
            return_tensors='pt'
        ).input_ids.squeeze()
        
        # Replace pad_token_id with -100 so loss ignores padding
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        
        return {'pixel_values': pixel_values, 'labels': labels}

print('[SUCCESS] Custom OFPHIL OCR PyTorch Dataset class compiled successfully!')

[SUCCESS] Custom OFPHIL OCR PyTorch Dataset class compiled successfully!


In [4]:
# ==============================================================================
# Step 4: Train / Validation Split (Stratified 85/15) & Fast DataLoader Setup
# ==============================================================================
import logging
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

from sklearn.model_selection import train_test_split
from transformers import AutoImageProcessor, RobertaTokenizer, TrOCRProcessor

# 1. Stratified 85/15 Split
train_df, val_df = train_test_split(valid_df, test_size=0.15, random_state=42, stratify=valid_df['box_type'])
print(f'Training Samples  : {len(train_df)}')
print(f'Validation Samples: {len(val_df)}')

# 2. Fast Processor Setup
model_id = 'microsoft/trocr-base-handwritten'
image_processor = AutoImageProcessor.from_pretrained(model_id)
tokenizer = RobertaTokenizer.from_pretrained(model_id)
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

# 3. Create PyTorch Datasets & Accelerated DataLoaders (BATCH_SIZE = 4 for safe 4GB GPU memory)
BATCH_SIZE = 4
train_dataset = OFPHILOCRDataset(train_df, PROJECT_ROOT, processor)
val_dataset = OFPHILOCRDataset(val_df, PROJECT_ROOT, processor)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    pin_memory=(device.type == 'cuda'),
    num_workers=0
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    pin_memory=(device.type == 'cuda'),
    num_workers=0
)

print(f'[SUCCESS] Robust DataLoaders ready: {len(train_loader)} batches (batch_size={BATCH_SIZE}) on {device}.')

Training Samples  : 18604
Validation Samples: 3284
[SUCCESS] Robust DataLoaders ready: 4651 batches (batch_size=4) on cuda.


In [5]:
# ==============================================================================
# Step 5: TrOCR Continuation Pipeline (Epochs 4 & 5)
# Starting Checkpoint: checkpoint_epoch_3
# Encoder: 100% Frozen | Decoder LR: 1e-5 | Fresh Optimizer & Scheduler
# Best Model Baseline: Epoch 2 (CER: 0.3374)
# ==============================================================================
import os
import sys
import time
import json
import torch
import gc
import re
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    VisionEncoderDecoderModel,
    AutoImageProcessor,
    RobertaTokenizer,
    TrOCRProcessor,
    get_linear_schedule_with_warmup,
    logging as hf_logging
)

hf_logging.set_verbosity_error()

# Clean residual GPU cache before starting
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ocr.domain_dictionary import DomainDictionary
dict_engine = DomainDictionary()

MODEL_CHECKPOINT_BASE = PROJECT_ROOT / "models" / "trocr_progressive_checkpoints"
MODEL_CHECKPOINT_BASE.mkdir(parents=True, exist_ok=True)
BEST_MODEL_DIR = PROJECT_ROOT / "models" / "trocr_ofphil_finetuned"
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUT_BENCHMARK_DIR = PROJECT_ROOT / "data" / "interim" / "document_processing" / "model_predictions_and_benchmarks"
OUT_BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

STARTING_CHECKPOINT = MODEL_CHECKPOINT_BASE / "checkpoint_epoch_3"
assert STARTING_CHECKPOINT.exists(), f"Missing checkpoint: {STARTING_CHECKPOINT}"

# Evaluation Helpers
def normalize_eval_text(text):
    if not text: return ""
    return re.sub(r'\s+', ' ', str(text).strip().upper())

def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if len(s2) == 0: return len(s1)
    prev = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            ins = prev[j + 1] + 1
            dels = curr[j] + 1
            subs = prev[j] + (c1 != c2)
            curr.append(min(ins, dels, subs))
        prev = curr
    return prev[-1]

def calc_cer(ref, hyp):
    r = normalize_eval_text(ref)
    h = normalize_eval_text(hyp)
    if not r: return 0.0 if not h else 1.0
    return min(1.0, levenshtein(r, h) / len(r))

def calc_wer(ref, hyp):
    rw = normalize_eval_text(ref).split()
    hw = normalize_eval_text(hyp).split()
    if not rw: return 0.0 if not hw else 1.0
    return min(1.0, levenshtein(rw, hw) / len(rw))

def run_dual_validation(model, processor, val_df, project_root, device):
    model.eval()
    results = []
    with torch.no_grad():
        for _, row in val_df.iterrows():
            img_p = project_root / row['relative_path']
            bx, by = int(row['bbox_x']), int(row['bbox_y'])
            bw, bh = int(row['bbox_width']), int(row['bbox_height'])
            gt_text = str(row['ground_truth_transcription']).strip() if pd.notna(row['ground_truth_transcription']) else ""
            box_type = row.get('box_type', 'UNKNOWN')
            
            try:
                with Image.open(img_p).convert('RGB') as full_img:
                    crop = full_img.crop((bx, by, bx + bw, by + bh))
                    pixel_values = processor(crop, return_tensors='pt').pixel_values.to(device)
                
                gen_ids = model.generate(pixel_values, max_new_tokens=64, num_beams=1)
                pred_raw = processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()
                if dict_engine and hasattr(dict_engine, 'correct_word'):
                    words = pred_raw.split()
                    pred_dict = " ".join([dict_engine.correct_word(w)[0] for w in words])
                else:
                    pred_dict = pred_raw
            except Exception:
                pred_raw = ""
                pred_dict = ""
                
            results.append({
                'relative_path': str(row['relative_path']),
                'box_type': box_type,
                'ground_truth': gt_text,
                'pred_raw': pred_raw,
                'pred_dict': pred_dict,
                'cer_raw': calc_cer(gt_text, pred_raw),
                'wer_raw': calc_wer(gt_text, pred_raw),
                'cer_dict': calc_cer(gt_text, pred_dict),
                'wer_dict': calc_wer(gt_text, pred_dict)
            })
            
    df_res = pd.DataFrame(results)
    raw_cer = float(df_res['cer_raw'].mean())
    raw_wer = float(df_res['wer_raw'].mean())
    raw_acc = (1.0 - raw_cer) * 100.0
    
    dict_cer = float(df_res['cer_dict'].mean())
    dict_wer = float(df_res['wer_dict'].mean())
    dict_acc = (1.0 - dict_cer) * 100.0
    
    return raw_cer, raw_wer, raw_acc, dict_cer, dict_wer, dict_acc, df_res

# ------------------------------------------------------------------------------
# LOAD WEIGHTS FROM CHECKPOINT 3 & FREEZE ENCODER
# ------------------------------------------------------------------------------
print(f"--> Loading weights from '{STARTING_CHECKPOINT}'...", flush=True)
processor = TrOCRProcessor.from_pretrained(STARTING_CHECKPOINT)
model = VisionEncoderDecoderModel.from_pretrained(STARTING_CHECKPOINT).to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

try:
    model.encoder.gradient_checkpointing_enable()
    print("--> [SPEED & MEMORY OPTIMIZATION] Gradient Checkpointing ENABLED on ViT Encoder.", flush=True)
except Exception as e:
    print(f"--> Gradient Checkpointing note: {e}", flush=True)

# FREEZE ENTIRE ViT ENCODER
for p in model.encoder.parameters(): p.requires_grad = False
for p in model.decoder.parameters(): p.requires_grad = True

trainable_encoder = sum(p.numel() for p in model.encoder.parameters() if p.requires_grad)
trainable_decoder = sum(p.numel() for p in model.decoder.parameters() if p.requires_grad)
print(f"  Encoder Trainable Params: {trainable_encoder} (100% FROZEN)")
print(f"  Decoder Trainable Params: {trainable_decoder:,} (TRAINABLE)")

# FRESH OPTIMIZER & SCHEDULER (EPOCHS 4 & 5)
DECODER_LR = 1e-5
GRADIENT_ACCUMULATION_STEPS = 2
steps_per_epoch = len(train_loader)
opt_steps = (steps_per_epoch // GRADIENT_ACCUMULATION_STEPS) * 2
warmup_steps = int(opt_steps * 0.05)

optimizer = AdamW([{"params": model.decoder.parameters(), "lr": DECODER_LR, "weight_decay": 0.01}])
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=opt_steps)
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

# HISTORY PRESERVATION (EPOCHS 1-3)
history = {
    'epoch': [1, 2, 3],
    'stage': ["Stage 1 (Frozen Encoder)", "Stage 1 (Frozen Encoder)", "Stage 2 (Unfrozen Top 2 Layers)"],
    'train_loss': [2.3285, 1.1519, 1.2580],
    'val_loss': [1.4890, 1.1086, 1.2759],
    'val_cer_raw': [0.3810, 0.3374, 0.3758],
    'val_wer_raw': [0.5172, 0.4412, 0.4978],
    'val_char_acc_raw': [61.90, 66.26, 62.42],
    'val_cer_dict': [0.3810, 0.3374, 0.3758],
    'val_char_acc_dict': [61.90, 66.26, 62.42],
    'vram_peak_mb': [5696, 5696, 6098],
    'epoch_time_min': [175.3, 154.7, 233.8]
}

BEST_BENCHMARK_CER = 0.3374  # Epoch 2 Baseline
best_val_cer = BEST_BENCHMARK_CER
best_epoch = 2
print(f"--> Best Checkpoint Baseline: Epoch {best_epoch} (CER: {best_val_cer:.4f})")
print(f"--> Rule: Checkpoint will NOT be overwritten unless Val CER is strictly < {BEST_BENCHMARK_CER:.4f}\n")

# RUN EPOCHS 4 & 5
for epoch in [4, 5]:
    epoch_start_time = time.time()
    step_time = time.time()
    model.train()
    total_train_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    
    stage_desc = f"Continuation (Decoder Refinement, LR: {DECODER_LR})"
    print(f"\n{'='*75}")
    print(f" >>> Epoch [{epoch}/5] Starting ({stage_desc})...")
    print(f"{'='*75}")
    
    for step, batch in enumerate(train_loader, start=1):
        pixel_values = batch['pixel_values'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        
        if scaler:
            with torch.amp.autocast('cuda'):
                outputs = model(pixel_values=pixel_values, labels=labels)
                loss = outputs.loss / GRADIENT_ACCUMULATION_STEPS
            scaler.scale(loss).backward()
            
            if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
        else:
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss / GRADIENT_ACCUMULATION_STEPS
            loss.backward()
            
            if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                
        total_train_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
        
        # Log progress every 100 steps
        if step % 100 == 0 or step == len(train_loader):
            elapsed = time.time() - step_time
            step_time = time.time()
            steps_left = len(train_loader) - step
            batch_speed = elapsed / 100 if step % 100 == 0 else elapsed
            eta_mins = (steps_left * batch_speed) / 60
            vram_mb = torch.cuda.memory_reserved(0) / (1024**2) if torch.cuda.is_available() else 0
            cur_loss = total_train_loss / step
            
            print(f"  Step [{step}/{len(train_loader)}] ({step/len(train_loader)*100:.1f}%) | "
                  f"Train Loss: {cur_loss:.4f} | ETA: {eta_mins:.1f}m | VRAM: {vram_mb:.0f} MB", flush=True)
            sys.stdout.flush()
            
        if step % 250 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation Phase
    model.eval()
    total_val_loss = 0.0
    print(f"\n  Running Validation for Epoch {epoch}...", flush=True)
    with torch.no_grad():
        for batch in val_loader:
            pixel_values = batch['pixel_values'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            if scaler:
                with torch.amp.autocast('cuda'):
                    outputs = model(pixel_values=pixel_values, labels=labels)
            else:
                outputs = model(pixel_values=pixel_values, labels=labels)
            total_val_loss += outputs.loss.item()
    avg_val_loss = total_val_loss / len(val_loader)
    
    # Dual Validation: Raw vs DomainDictionary
    (raw_cer, raw_wer, raw_acc,
     dict_cer, dict_wer, dict_acc,
     df_epoch_preds) = run_dual_validation(model, processor, val_df, PROJECT_ROOT, device)
     
    duration_min = (time.time() - epoch_start_time) / 60
    vram_peak = torch.cuda.max_memory_reserved(0) / (1024**2) if torch.cuda.is_available() else 0
    
    print("-" * 75, flush=True)
    print(f" [EPOCH {epoch} DONE] Time: {duration_min:.1f} min | VRAM Peak: {vram_peak:.0f} MB", flush=True)
    print(f" Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}", flush=True)
    print(f" Raw TrOCR           : Val CER = {raw_cer:.4f} ({raw_cer*100:.2f}%) | Val WER = {raw_wer:.4f} | Accuracy = {raw_acc:.2f}%", flush=True)
    print(f" TrOCR + Dictionary  : Val CER = {dict_cer:.4f} ({dict_cer*100:.2f}%) | Val WER = {dict_wer:.4f} | Accuracy = {dict_acc:.2f}%", flush=True)
    print("-" * 75, flush=True)
    
    # Record history
    history['epoch'].append(epoch)
    history['stage'].append(stage_desc)
    history['train_loss'].append(round(avg_train_loss, 4))
    history['val_loss'].append(round(avg_val_loss, 4))
    history['val_cer_raw'].append(round(raw_cer, 4))
    history['val_wer_raw'].append(round(raw_wer, 4))
    history['val_char_acc_raw'].append(round(raw_acc, 2))
    history['val_cer_dict'].append(round(dict_cer, 4))
    history['val_char_acc_dict'].append(round(dict_acc, 2))
    history['vram_peak_mb'].append(round(vram_peak, 0))
    history['epoch_time_min'].append(round(duration_min, 1))
    
    # Save checkpoint
    epoch_ckpt_dir = MODEL_CHECKPOINT_BASE / f"checkpoint_epoch_{epoch}"
    epoch_ckpt_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(epoch_ckpt_dir)
    processor.save_pretrained(epoch_ckpt_dir)
    print(f"  --> [SAVED CHECKPOINT] {epoch_ckpt_dir}", flush=True)
    
    # Evaluate against Best Model Rule
    primary_eval_cer = dict_cer
    if primary_eval_cer < best_val_cer:
        print(f"  --> [NEW BEST MODEL] CER improved from {best_val_cer:.4f} to {primary_eval_cer:.4f}!", flush=True)
        best_val_cer = primary_eval_cer
        best_epoch = epoch
        model.save_pretrained(BEST_MODEL_DIR)
        processor.save_pretrained(BEST_MODEL_DIR)
        print(f"  --> [UPDATED BEST MODEL] Saved to {BEST_MODEL_DIR} (Epoch {best_epoch}, CER: {best_val_cer:.4f})\n", flush=True)
    else:
        print(f"  --> [RETAINED BEST MODEL] Best remains Epoch {best_epoch} (CER: {best_val_cer:.4f} <= Epoch {epoch} CER {primary_eval_cer:.4f})\n", flush=True)
        
    pd.DataFrame(history).to_csv(OUT_BENCHMARK_DIR / "trocr_5epoch_training_history.csv", index=False)
    
    # Guard check for Epoch 4
    if epoch == 4 and primary_eval_cer > 0.45:
        print("\n[WARNING] Validation CER deteriorated above 0.45. Pausing before Epoch 5.", flush=True)
        break

# Final Independent Test Evaluation (Untouched Test Partition)
print("\n" + "=" * 75, flush=True)
print(f" Running Final Evaluation on UNTOUCHED Test Partition using Best Model (Epoch {best_epoch})", flush=True)
print("=" * 75, flush=True)
best_eval_model = VisionEncoderDecoderModel.from_pretrained(BEST_MODEL_DIR).to(device)
best_eval_model.eval()

split_test_csv = PROJECT_ROOT / "data" / "interim" / "document_processing" / "splits" / "test_unseen_documents_crops.csv"
eval_test_df = pd.read_csv(split_test_csv) if split_test_csv.exists() else val_df

(test_raw_cer, test_raw_wer, test_raw_acc,
 test_dict_cer, test_dict_wer, test_dict_acc,
 df_final_preds) = run_dual_validation(best_eval_model, processor, eval_test_df, PROJECT_ROOT, device)

print(f"Test Partition Samples Evaluated : {len(df_final_preds)}", flush=True)
print(f"Raw TrOCR Test CER               : {test_raw_cer:.4f} ({test_raw_cer*100:.2f}%) | Accuracy: {test_raw_acc:.2f}%", flush=True)
print(f"TrOCR + Dict Test CER            : {test_dict_cer:.4f} ({test_dict_cer*100:.2f}%) | Accuracy: {test_dict_acc:.2f}%", flush=True)
print(f"Final Test WER                   : {test_dict_wer:.4f} ({test_dict_wer*100:.2f}%)", flush=True)
print("=" * 75, flush=True)

df_final_preds.to_csv(OUT_BENCHMARK_DIR / "trocr_finetuned_benchmark.csv", index=False)
with open(OUT_BENCHMARK_DIR / "trocr_finetuned_evaluation_report.json", 'w', encoding='utf-8') as f:
    json.dump({
        'model_name': 'microsoft/trocr-base-handwritten (OFPHIL Progressive Continuation)',
        'best_epoch_selected': best_epoch,
        'best_validation_cer': best_val_cer,
        'final_test_cer_raw': test_raw_cer,
        'final_test_char_acc_raw': test_raw_acc,
        'final_test_cer_with_dict': test_dict_cer,
        'final_test_char_acc_with_dict': test_dict_acc,
        'final_test_wer': test_dict_wer,
        'per_epoch_history': history,
        'baseline_comparison': {
            'original_epoch3_baseline_cer': 0.3685,
            'epoch2_best_cer': 0.3374,
            'achieved_best_cer': best_val_cer
        }
    }, f, indent=4)

print("[COMPLETE] Weight-Based Continuation for Epochs 4 & 5 Finished Successfully!", flush=True)

--> Loading weights from 'c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\dss-charity-eligibity-classification\models\trocr_progressive_checkpoints\checkpoint_epoch_3'...


Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

--> [SPEED & MEMORY OPTIMIZATION] Gradient Checkpointing ENABLED on ViT Encoder.
  Encoder Trainable Params: 0 (100% FROZEN)
  Decoder Trainable Params: 247,268,352 (TRAINABLE)
--> Best Checkpoint Baseline: Epoch 2 (CER: 0.3374)
--> Rule: Checkpoint will NOT be overwritten unless Val CER is strictly < 0.3374


 >>> Epoch [4/5] Starting (Continuation (Decoder Refinement, LR: 1e-05))...
  Step [100/4651] (2.2%) | Train Loss: 0.9327 | ETA: 171.1m | VRAM: 5632 MB
  Step [200/4651] (4.3%) | Train Loss: 0.8662 | ETA: 167.6m | VRAM: 5632 MB
  Step [300/4651] (6.5%) | Train Loss: 0.8967 | ETA: 163.2m | VRAM: 5692 MB
  Step [400/4651] (8.6%) | Train Loss: 0.8552 | ETA: 157.2m | VRAM: 5692 MB
  Step [500/4651] (10.8%) | Train Loss: 0.8343 | ETA: 154.8m | VRAM: 5692 MB
  Step [600/4651] (12.9%) | Train Loss: 0.8008 | ETA: 126.2m | VRAM: 5696 MB
  Step [700/4651] (15.1%) | Train Loss: 0.8047 | ETA: 120.3m | VRAM: 5696 MB
  Step [800/4651] (17.2%) | Train Loss: 0.7943 | ETA: 116.6m | VRAM: 5676 MB


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  --> [SAVED CHECKPOINT] c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\dss-charity-eligibity-classification\models\trocr_progressive_checkpoints\checkpoint_epoch_4
  --> [NEW BEST MODEL] CER improved from 0.3374 to 0.3263!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  --> [UPDATED BEST MODEL] Saved to c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\dss-charity-eligibity-classification\models\trocr_ofphil_finetuned (Epoch 5, CER: 0.3193)


 Running Final Evaluation on UNTOUCHED Test Partition using Best Model (Epoch 5)


Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

Test Partition Samples Evaluated : 2342
Raw TrOCR Test CER               : 0.2549 (25.49%) | Accuracy: 74.51%
TrOCR + Dict Test CER            : 0.2567 (25.67%) | Accuracy: 74.33%
Final Test WER                   : 0.3310 (33.10%)
[COMPLETE] Weight-Based Continuation for Epochs 4 & 5 Finished Successfully!
